# 07 · Extracción estructurada de necropsias con LLM

**Proyecto Integrador MNA — Cotejo ante-mortem / post-mortem (Querétaro)**

**Objetivo.** Llevar los registros de necropsias al MISMO esquema estructurado que ya tienen las fichas de búsqueda
(modelos Pydantic, vocabularios controlados, texto canónico por faceta), para poder comparar prenda contra prenda
y tatuaje contra tatuaje en la fase de cotejo.

**Entrada.** Texto descriptivo de las necropsias (ropa, pertenencias, tatuajes, vestimenta, cicatrices) — sin imágenes.
**Salida.** Tablas silver en Unity Catalog (`proyecto_mna.fichas_busqueda`) listas para embeddings y Vector Search.

In [0]:
%run ./00_config_evaluacion

In [0]:
# Instalación de openai (no está pre-instalado en serverless)
%pip install -q openai
dbutils.library.restartPython()

In [0]:
%run ./00_config_evaluacion

In [0]:
# ── Widgets propios de este notebook ──
dbutils.widgets.text("endpoint", "proyecto_mna.fichas_busqueda.gema", "Modelo (AI Gateway)")
dbutils.widgets.text("limite", "20", "Límite de filas (0 = todas)")
dbutils.widgets.text("max_workers", "4", "Llamadas concurrentes")
dbutils.widgets.dropdown("sobrescribir", "false", ["true", "false"], "Sobrescribir tablas existentes")
dbutils.widgets.dropdown("strict_schema", "false", ["true", "false"], "Forzar JSON schema estricto")

ENDPOINT = dbutils.widgets.get("endpoint")
LIMITE = int(dbutils.widgets.get("limite"))
MAX_WORKERS = int(dbutils.widgets.get("max_workers"))
SOBRESCRIBIR = dbutils.widgets.get("sobrescribir") == "true"
STRICT_SCHEMA = dbutils.widgets.get("strict_schema") == "true"

print(f"Endpoint: {ENDPOINT} | Límite: {LIMITE} | Workers: {MAX_WORKERS} | Sobrescribir: {SOBRESCRIBIR}")

## 1. Esquema Pydantic

Modelos copiados de `01_extraccion_fichas_llm` (Prenda, Tatuaje, SenaParticular) sin cambiar sus vocabularios.
Se agrega el modelo **Pertenencia** y el modelo de salida **NecropsiaExtraida**.

In [0]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field

SIN_DATO = "SIN_DATO"

# ── Enums copiados del notebook 01 (sin cambios) ──
Region = Literal["CABEZA", "CARA", "CUELLO", "HOMBRO", "PECHO", "ABDOMEN", "ESPALDA", "BRAZO",
                 "ANTEBRAZO", "MUNECA", "MANO", "DEDOS", "CADERA", "GLUTEO", "MUSLO", "RODILLA",
                 "PIERNA", "TOBILLO", "PIE", "OTRA", "SIN_DATO"]
Lado = Literal["IZQUIERDO", "DERECHO", "CENTRAL", "BILATERAL", "SIN_DATO"]
Origen = Literal["TEXTO_FICHA", "FOTO", "AMBOS"]
CategoriaTatuaje = Literal["NOMBRE_O_TEXTO", "FECHA_O_NUMERO", "RELIGIOSO", "ANIMAL", "FLORA",
                           "CALAVERA", "FIGURA_HUMANA", "PERSONAJE", "SIMBOLO", "TRIBAL_ABSTRACTO",
                           "OTRO", "SIN_DATO"]
ColorTinta = Literal["NEGRO_GRIS", "COLOR", "TINTA_BLANCA", "SIN_DATO"]
TipoSena = Literal["CICATRIZ", "LUNAR", "MANCHA", "PERFORACION", "AMPUTACION", "MALFORMACION",
                   "PROTESIS_O_IMPLANTE", "CIRUGIA", "RASGO_DENTAL", "OTRA"]
TipoPrenda = Literal["CAMISA", "PLAYERA", "BLUSA", "SUDADERA", "CHAMARRA", "CHALECO", "SUETER",
                     "PANTALON", "SHORT", "FALDA", "VESTIDO", "ROPA_INTERIOR", "CALZADO",
                     "GORRA_O_SOMBRERO", "ACCESORIO", "OTRA"]

# ── Nuevo enum para pertenencias ──
TipoPertenencia = Literal["DOCUMENTO", "CELULAR", "JOYERIA", "RELOJ", "CARTERA", "MOCHILA",
                          "DINERO", "LLAVES", "OTRA", "SIN_DATO"]


# ── Modelos copiados del notebook 01 (sin cambios) ──
class Tatuaje(BaseModel):
    region: Region
    lado: Lado
    categoria: CategoriaTatuaje
    motivo: str = Field(description="Descripción breve del diseño, máx. 20 palabras")
    texto_legible: Optional[str] = Field(None, description="Letras/números del tatuaje tal cual, si los hay")
    color_tinta: ColorTinta = SIN_DATO
    tamano_cm: Optional[float] = None
    origen: Origen


class SenaParticular(BaseModel):
    tipo: TipoSena
    region: Region
    lado: Lado
    descripcion: str = Field(description="Descripción breve, máx. 20 palabras")
    tamano_cm: Optional[float] = None
    origen: Origen


class Prenda(BaseModel):
    tipo: TipoPrenda
    color: str = Field(description="Color principal en minúsculas, o 'sin_dato'")
    descripcion: Optional[str] = Field(None, description="Estampado, material, logotipos; máx. 15 palabras")
    marca: Optional[str] = None
    origen: Origen


# ── Nuevo modelo Pertenencia ──
class Pertenencia(BaseModel):
    tipo: TipoPertenencia
    descripcion: str = Field(description="Descripción breve, máx. 15 palabras")
    marca: Optional[str] = None


# ── Modelo de salida para necropsias ──
class NecropsiaExtraida(BaseModel):
    prendas: List[Prenda]
    tatuajes: List[Tatuaje]
    senas: List[SenaParticular]
    pertenencias: List[Pertenencia]
    notas_extraccion: Optional[str] = Field(None, description="Ambigüedades o datos incompletos")


def json_schema_estricto(model) -> dict:
    """Schema JSON apto para response_format strict: sin defaults y con todos los campos requeridos."""
    schema = model.model_json_schema()

    def _fix(node):
        if isinstance(node, dict):
            if node.get("type") == "object" and "properties" in node:
                node["required"] = list(node["properties"].keys())
                node["additionalProperties"] = False
            node.pop("default", None)
            node.pop("title", None)
            for v in node.values():
                _fix(v)
        elif isinstance(node, list):
            for v in node:
                _fix(v)

    _fix(schema)
    return schema


SCHEMA_JSON = json_schema_estricto(NecropsiaExtraida)
print(f"Campos de primer nivel: {len(SCHEMA_JSON['properties'])}")

In [0]:
import unicodedata

# Sinónimos frecuentes en fichas/necropsias -> valor del vocabulario controlado.
# Se aplican solo si el resultado es válido para ESE campo.
SINONIMOS = {
    # cabello
    "LISO": "LACIO", "CHINO": "RIZADO", "QUEBRADO": "ONDULADO", "CALVO": "SIN_CABELLO",
    "CANOSO": "CANO", "BLANCO": "CANO", "CAFE": "CASTANO_OSCURO", "CASTANO": "CASTANO_OSCURO",
    "PINTADO": "TENIDO", "TENIDO": "TENIDO", "PELON": "RAPADO",
    # ojos
    "CAFES": "CAFES_OSCUROS", "CAFE_OSCURO": "CAFES_OSCUROS", "CAFE_CLARO": "CAFES_CLAROS",
    "CHICOS": "CHICOS", "PEQUENOS": "CHICOS", "CHICA": "PEQUENA", "CHICO": "PEQUENA",
    # piel / cara / nariz
    "TRIGUENO": "MORENO_CLARO", "APIÑONADO": "MORENO_CLARO", "APINONADO": "MORENO_CLARO",
    "OVALO": "OVALADA", "OVAL": "OVALADA", "REDONDO": "REDONDA", "CHATO": "CHATA",
    "GRUESA": "ANCHA", "RESPINGADO": "RESPINGADA", "DELGADO": "DELGADA",
    # región corporal
    "MEJILLA": "CARA", "POMULO": "CARA", "FRENTE": "CARA", "CEJA": "CARA", "OJO": "CARA",
    "LABIO": "CARA", "MENTON": "CARA", "NARIZ": "CARA", "OREJA": "CABEZA", "NUCA": "CUELLO",
    "CUERO_CABELLUDO": "CABEZA", "TORAX": "PECHO", "SENO": "PECHO", "CLAVICULA": "PECHO",
    "ESTOMAGO": "ABDOMEN", "VIENTRE": "ABDOMEN", "OMBLIGO": "ABDOMEN", "COSTADO": "ABDOMEN",
    "COSTILLA": "ABDOMEN", "OMOPLATO": "ESPALDA", "ESPALDA_BAJA": "ESPALDA", "BICEP": "BRAZO",
    "BICEPS": "BRAZO", "CODO": "BRAZO", "PALMA": "MANO", "PORLICUE": "MANO", "NUDILLOS": "DEDOS",
    "DEDO": "DEDOS", "NALGA": "GLUTEO", "PANTORRILLA": "PIERNA", "PANTORILLA": "PIERNA",
    "CHAMORRO": "PIERNA", "ESPINILLA": "PIERNA", "TALON": "PIE", "INGLE": "CADERA",
    # lado
    "AMBOS": "BILATERAL", "AMBOS_LADOS": "BILATERAL", "CENTRO": "CENTRAL", "MEDIO": "CENTRAL",
    # vestimenta
    "TENIS": "CALZADO", "ZAPATOS": "CALZADO", "ZAPATO": "CALZADO", "BOTAS": "CALZADO",
    "HUARACHES": "CALZADO", "SANDALIAS": "CALZADO", "CHANCLAS": "CALZADO", "JEANS": "PANTALON",
    "PANTALON_DE_MEZCLILLA": "PANTALON", "MEZCLILLA": "PANTALON", "PANTS": "PANTALON",
    "BERMUDA": "SHORT", "SHORTS": "SHORT", "POLO": "PLAYERA", "CAMISETA": "PLAYERA",
    "CHAQUETA": "CHAMARRA", "ABRIGO": "CHAMARRA", "JERSEY": "SUETER", "SUETER": "SUETER",
    "GORRA": "GORRA_O_SOMBRERO", "SOMBRERO": "GORRA_O_SOMBRERO", "CACHUCHA": "GORRA_O_SOMBRERO",
    "CINTURON": "ACCESORIO", "COLLAR": "ACCESORIO", "PULSERA": "ACCESORIO", "RELOJ": "ACCESORIO",
    "ANILLO": "ACCESORIO", "ARETES": "ACCESORIO", "LENTES": "ACCESORIO", "MOCHILA": "ACCESORIO",
    "BOXER": "ROPA_INTERIOR", "CALZON": "ROPA_INTERIOR", "BRASIER": "ROPA_INTERIOR",
    "SOSTEN": "ROPA_INTERIOR", "CALCETINES": "ROPA_INTERIOR",
    # tatuajes
    "LETRAS": "NOMBRE_O_TEXTO", "LETRA": "NOMBRE_O_TEXTO", "TEXTO": "NOMBRE_O_TEXTO",
    "NOMBRE": "NOMBRE_O_TEXTO", "FRASE": "NOMBRE_O_TEXTO", "FECHA": "FECHA_O_NUMERO",
    "NUMEROS": "FECHA_O_NUMERO", "NUMERO": "FECHA_O_NUMERO", "CRUZ": "RELIGIOSO",
    "VIRGEN": "RELIGIOSO", "ROSARIO": "RELIGIOSO", "SANTA_MUERTE": "RELIGIOSO", "ROSA": "FLORA",
    "FLOR": "FLORA", "CRANEO": "CALAVERA", "ESTRELLA": "SIMBOLO", "CORAZON": "SIMBOLO",
    "TRIBAL": "TRIBAL_ABSTRACTO", "ABSTRACTO": "TRIBAL_ABSTRACTO", "NEGRO": "NEGRO_GRIS",
    "GRIS": "NEGRO_GRIS", "NEGRA": "NEGRO_GRIS", "COLORES": "COLOR", "A_COLOR": "COLOR",
    "BLANCA": "TINTA_BLANCA",
    # genéricos de faltante
    "NO_ESPECIFICADO": "SIN_DATO", "NINGUNO": "SIN_DATO", "DESCONOCIDO": "SIN_DATO",
    "NO_VISIBLE": "SIN_DATO", "N/A": "SIN_DATO", "NA": "SIN_DATO", "NULL": "SIN_DATO",
}


def _canon(txt: str) -> str:
    t = unicodedata.normalize("NFKD", str(txt)).encode("ascii", "ignore").decode().upper().strip()
    return "_".join(t.replace("-", " ").replace("/", " ").split())


def _variantes(v: str):
    """Variantes de género/número (IZQUIERDA->IZQUIERDO, GRANDE->GRANDES) y su sinónimo."""
    formas = [v, v + "S", v + "ES"]
    if v.endswith("S"):
        formas.append(v[:-1])
    if v[-1:] in "AO":
        swap = v[:-1] + ("O" if v[-1] == "A" else "A")
        formas += [swap, swap + "S"]
    for f in formas:
        yield f
        if f in SINONIMOS:
            yield SINONIMOS[f]


def _norm_enum(valor, enum, ruta, log):
    if valor is None:
        return "SIN_DATO" if "SIN_DATO" in enum else valor
    v = _canon(valor)
    for cand in _variantes(v):
        if cand in enum:
            if cand != v:
                log.append(f"{ruta}: {valor}->{cand}")
            return cand
    destino = "OTRA" if "OTRA" in enum else ("SIN_DATO" if "SIN_DATO" in enum else enum[0])
    log.append(f"{ruta}: {valor}->{destino} (fuera de vocabulario)")
    return destino


def _resolver(s):
    return SCHEMA_JSON["$defs"][s["$ref"].split("/")[-1]] if "$ref" in s else s


def _norm(valor, s, ruta, log):
    s = _resolver(s)
    if "anyOf" in s:
        if valor is None:
            return None
        no_nulos = [_resolver(o) for o in s["anyOf"] if o.get("type") != "null"]
        s = no_nulos[0] if no_nulos else s
    if "enum" in s:
        return _norm_enum(valor, s["enum"], ruta, log)
    if s.get("type") == "object" and isinstance(valor, dict):
        props = s.get("properties", {})
        return {k: (_norm(v, props[k], f"{ruta}.{k}".lstrip("."), log) if k in props else v)
                for k, v in valor.items()}
    if s.get("type") == "array" and isinstance(valor, list):
        return [_norm(v, s["items"], f"{ruta}[{i}]", log) for i, v in enumerate(valor)]
    return valor


def normalizar_salida(datos: dict) -> tuple[dict, list[str]]:
    """Ajusta la salida del LLM al vocabulario controlado y devuelve la lista de cambios aplicados."""
    log: list[str] = []
    return _norm(datos, SCHEMA_JSON, "", log), log

## 2. Prompt, cliente LLM y función de extracción

El prompt es específico para necropsias: entrada de **texto** etiquetado (no imágenes),
extrayendo vestimenta, pertenencias, tatuajes y señas. La huella del prompt + esquema
drive la idempotencia: si cambian, se reprocesa todo automáticamente.

In [0]:
import hashlib
import json
import time
from datetime import datetime, timezone

from openai import OpenAI
from pydantic import ValidationError

# ── Prompt específico para extracción de necropsias (texto, no imágenes) ──
SYSTEM_PROMPT = """Eres un asistente de extracción de datos para peritos forenses en México.
Recibes TEXTO descriptivo del registro de una necropsia (no imágenes) sobre vestimenta, pertenencias,
tatuajes y señas particulares del cuerpo recibido en el Servicio Médico Forense.
Devuelves ÚNICAMENTE un JSON que cumple el esquema dado.

Reglas:
1. El texto de entrada tiene secciones etiquetadas (ROPA VESTIDA, PERTENENCIAS, DESCRIPCIÓN DE TATUAJES,
   DESCRIPCIÓN DE VESTIMENTA, ¿TIENE CICATRIZ?). Si una sección no aparece o está vacía, no extraigas nada de ella.
2. No infieras ni adivines. Si un dato no se menciona explícitamente, usa SIN_DATO o deja la lista vacía.
3. NO transcribas nombres de personas, folios ni carpetas de investigación en ningún campo.
4. VESTIMENTA: un elemento Prenda por cada prenda mencionada. Si el texto dice "playera azul" crea
   una prenda tipo PLAYERA, color "azul". Si no hay descripción de ropa, lista vacía.
5. PERTENENCIAS: un elemento Pertenencia por cada objeto mencionado (documento, celular, joyería, etc.).
6. TATUAJES: un elemento Tatuaje por cada tatuaje descrito. Si solo dice "NO" o "SIN TATUAJES", lista vacía.
   Usa region y lado del vocabulario. En motivo describe el diseño brevemente.
7. SEÑAS: Si tiene_cicatriz dice "SI" pero no hay descripción de ubicación o características, NO crees una seña.
   Solo crea una seña si hay una descripción real (ubicación, tipo, tamaño).
8. El campo origen siempre debe ser "TEXTO_FICHA" (no hay foto).
9. Normaliza al vocabulario del esquema (ej. "tenis" -> CALZADO, "camiseta" -> PLAYERA).
"""

# Huella estable del prompt + esquema: si cambian, se reprocesa todo automáticamente.
_huella = hashlib.sha256((SYSTEM_PROMPT + json.dumps(SCHEMA_JSON, sort_keys=True)).encode()).hexdigest()[:8]
PROMPT_FINGERPRINT = f"nec_v1:{_huella}"
print("Prompt:", PROMPT_FINGERPRINT)

# ── Cliente OpenAI-compatible (AI Gateway) ──
_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
HOST = _ctx.apiUrl().get().rstrip("/")
client = OpenAI(base_url=f"{HOST}/ai-gateway/mlflow/v1", api_key=_ctx.apiToken().get())


def _formato_texto():
    """Formato de salida en la Responses API (parámetro `text`)."""
    if STRICT_SCHEMA:
        return {"format": {"type": "json_schema", "name": "necropsia_extraida",
                           "schema": SCHEMA_JSON, "strict": True}}
    return None


def _limpiar_json(texto: str) -> str:
    """Quita ``` y cualquier texto antes/después del objeto JSON."""
    t = (texto or "").strip()
    ini, fin = t.find("{"), t.rfind("}")
    return t[ini:fin + 1] if ini != -1 and fin > ini else t


def extraer_necropsia(no_nec: int, texto_entrada: str, llave: str, reintentos: int = 2) -> dict:
    """Llama al LLM con el texto de la necropsia y devuelve un dict para bronze."""
    instrucciones = SYSTEM_PROMPT
    if not STRICT_SCHEMA:
        instrucciones += ("\nResponde SOLO con un objeto JSON (sin texto adicional ni ```) que cumpla este esquema:\n"
                          + json.dumps(SCHEMA_JSON, ensure_ascii=False))
    entrada = [
        {"role": "system", "content": [{"type": "input_text", "text": instrucciones}]},
        {"role": "user", "content": [{"type": "input_text", "text": texto_entrada}]},
    ]
    kwargs = {"text": _formato_texto()} if STRICT_SCHEMA else {}
    base = {"no_nec": int(no_nec), "llave": llave, "modelo": ENDPOINT, "prompt": PROMPT_FINGERPRINT}
    ultimo_error = None
    for intento in range(1, reintentos + 1):
        t0 = time.time()
        try:
            resp = client.responses.create(
                model=ENDPOINT, input=entrada, temperature=0, max_output_tokens=4000, **kwargs,
            )
            datos = json.loads(_limpiar_json(resp.output_text))
            datos, cambios = normalizar_salida(datos)
            if cambios:
                datos["notas_extraccion"] = " | ".join(
                    filter(None, [datos.get("notas_extraccion"), "normalizado: " + "; ".join(cambios)]))
            nec = NecropsiaExtraida.model_validate(datos)
            return {**base, "ok": True, "error": None, "json": nec.model_dump_json(),
                    "latencia_s": round(time.time() - t0, 2), "intentos": intento,
                    "ts": datetime.now(timezone.utc).replace(tzinfo=None)}
        except ValidationError as e:
            ultimo_error = f"validacion: {e.errors()[:3]}"
            time.sleep(2 ** intento)
        except json.JSONDecodeError as e:
            ultimo_error = f"json_invalido: {e}"
            time.sleep(2 ** intento)
        except Exception as e:
            ultimo_error = f"{type(e).__name__}: {str(e)[:500]}"
            time.sleep(2 ** intento)
    return {**base, "ok": False, "error": ultimo_error, "json": None,
            "latencia_s": None, "intentos": reintentos, "ts": datetime.now(timezone.utc).replace(tzinfo=None)}

## 3. Conversión de edad y carga de datos

La columna `edad` viene como texto ("40-50 AÑOS", "43 AÑOS", "10-12 SDG"). Se convierte a `edad_min` y `edad_max`
con regex. Los valores en SDG/semanas/días/meses (fetos/recién nacidos) se descartan.

Se carga `T_NECROPSIAS` y se construye el texto de entrada etiquetado para el LLM.
Solo se incluyen las columnas permitidas (ropa, pertenencias, descr_tatuajes, descr_vestimenta, tiene_cicatriz).
NO se usan sintesis, resena_hecho ni causa_muerte (pueden contener nombres).

In [0]:
import re
import hashlib
import pandas as pd
from pyspark.sql import functions as F

# ── Función auxiliar para escribir tablas silver ──
def _escribir_tabla_pdf(pdf: pd.DataFrame, tabla: str):
    """Escribe un pandas DataFrame a tabla. Respeta el widget sobrescribir."""
    nombre = tabla.split('.')[-1]
    if not SOBRESCRIBIR and spark.catalog.tableExists(tabla):
        print(f"  {nombre}: ya existe (omitida, sobrescribir=false)")
        return
    if len(pdf) == 0:
        print(f"  {nombre}: sin filas")
        return
    spark.createDataFrame(pdf).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabla)
    print(f"  {nombre}: {len(pdf)} filas escritas")

# ── Conversión de edad (regex, no LLM) ──
_PAT_RANGE = re.compile(r'(\d+)\s*[-–Aa]\s*(\d+)\s*AÑOS', re.IGNORECASE)
_PAT_SINGLE = re.compile(r'(\d+)\s*AÑOS', re.IGNORECASE)

def convertir_edad(texto):
    """Convierte texto de edad ('40-50 AÑOS', '43 AÑOS') a (edad_min, edad_max).
    Devuelve (None, None) si no se puede convertir (SDG, SEMANAS, etc.)."""
    if not texto or not isinstance(texto, str):
        return (None, None)
    upper = texto.upper()
    if any(p in upper for p in ['SDG', 'SEMANA', 'DIA', 'MES']):
        return (None, None)
    m = _PAT_RANGE.search(texto)
    if m:
        return (int(m.group(1)), int(m.group(2)))
    m = _PAT_SINGLE.search(texto)
    if m:
        v = int(m.group(1))
        return (v, v)
    return (None, None)

# Pruebas sintéticas (no datos reales)
_pruebas = [("40-50 AÑOS", (40, 50)), ("43 AÑOS", (43, 43)), ("10-12 SDG", (None, None)),
            ("40 A 50 AÑOS", (40, 50)), ("50 AÑOS", (50, 50)), ("15.2 SDG", (None, None))]
for txt, esperado in _pruebas:
    real = convertir_edad(txt)
    assert real == esperado, f"convertir_edad('{txt}') = {real}, esperado {esperado}"
print("Pruebas de conversión de edad: OK")

# ── Cargar necropsias (solo columnas permitidas) ──
nec_df = (spark.table(T_NECROPSIAS)
          .select("no_nec", "sexo_norm", "edad", "fecha_ingreso", "tipo_ingreso", "status",
                  "ropa", "pertenencias", "descr_tatuajes", "descr_vestimenta", "tiene_cicatriz",
                  "municipio_hechos")
          .toPandas())

print(f"Necropsias cargadas: {len(nec_df)}")

# ── Construir texto de entrada etiquetado ──
_VACIOS = {"", "NO", "N/A", "SIN DATO", "SIN_DATO", "NINGUNO", "NINGUNA", "NULL", "NONE"}

def construir_texto_entrada(row):
    """Construye el texto etiquetado que se envía al LLM. Solo secciones no vacías."""
    secciones = []
    for etiqueta, col in [("ROPA VESTIDA", "ropa"), ("PERTENENCIAS", "pertenencias"),
                          ("DESCRIPCIÓN DE TATUAJES", "descr_tatuajes"),
                          ("DESCRIPCIÓN DE VESTIMENTA", "descr_vestimenta"),
                          ("¿TIENE CICATRIZ?", "tiene_cicatriz")]:
        val = row.get(col)
        if val is not None and str(val).strip().upper() not in _VACIOS:
            secciones.append(f"{etiqueta}: {val}")
    return "\n".join(secciones)

nec_df["texto_entrada"] = nec_df.apply(construir_texto_entrada, axis=1)
nec_df["llave"] = nec_df["texto_entrada"].apply(
    lambda t: hashlib.sha256(f"{t}|{ENDPOINT}|{PROMPT_FINGERPRINT}".encode()).hexdigest())

# Filtrar filas sin texto (nada que extraer)
nec_df = nec_df[nec_df["texto_entrada"].str.len() > 0].copy()

# Aplicar límite
if LIMITE > 0:
    nec_df = nec_df.head(LIMITE)

print(f"Necropsias con texto: {len(nec_df)}")

## 4. Extracción → bronze (idempotente)

Se procesan las necropsias pendientes con el LLM. La idempotencia se basa en la llave
(sha256 del texto de entrada + modelo + huella del prompt). Si `sobrescribir=true`, se trunca bronze primero.

In [0]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import Counter

# ── Crear tabla bronze si no existe ──
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {T_NEC_BRONZE} (
  no_nec BIGINT, llave STRING, modelo STRING, prompt STRING,
  json STRING, ok BOOLEAN, error STRING, latencia_s DOUBLE, intentos INT, ts TIMESTAMP
)
""")

# ── Si sobrescribir, limpiar bronze ──
if SOBRESCRIBIR:
    spark.sql(f"TRUNCATE TABLE {T_NEC_BRONZE}")
    print("Bronze truncado (sobrescribir=true)")

# ── Filas ya procesadas (idempotencia) ──
ya_procesadas = set(
    r["llave"] for r in spark.table(T_NEC_BRONZE)
    .filter(F.col("prompt") == PROMPT_FINGERPRINT)
    .filter(F.col("modelo") == ENDPOINT)
    .select("llave").distinct().collect()
)

pendientes = nec_df[~nec_df["llave"].isin(ya_procesadas)].copy()
print(f"Ya procesadas: {len(ya_procesadas)} | Pendientes: {len(pendientes)}")

# ── Extracción concurrente ──
resultados = []
if len(pendientes) > 0:
    filas = pendientes.to_dict("records")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {
            pool.submit(extraer_necropsia, f["no_nec"], f["texto_entrada"], f["llave"]): f
            for f in filas
        }
        for i, fut in enumerate(as_completed(futures), 1):
            resultados.append(fut.result())
            if i % 10 == 0 or i == len(filas):
                print(f"  {i}/{len(filas)}")

    # ── Guardar en bronze ──
    if resultados:
        spark.createDataFrame(pd.DataFrame(resultados)).write.mode("append").saveAsTable(T_NEC_BRONZE)

    # ── Resumen (solo agregados) ──
    n_ok = sum(1 for r in resultados if r["ok"])
    n_err = sum(1 for r in resultados if not r["ok"])
    print(f"Procesadas: {len(resultados)} | OK: {n_ok} | Errores: {n_err}")
    if n_err > 0:
        tipos_error = Counter(r["error"].split(":")[0] for r in resultados if not r["ok"])
        for tipo, n in tipos_error.items():
            print(f"  Error {tipo}: {n}")
else:
    print("No hay filas pendientes")

## 5. Aplanado → silver

A partir de los registros OK en bronze, se aplanan a tablas silver:
una fila por elemento (prenda, tatuaje, pertenencia) y una fila por necropsia (persona).
La edad se convierte con regex (no del LLM).

In [0]:
from pyspark.sql import Window

# ── Cargar bronze OK (dedup por llave, último ts) ──
bronze_rows = (spark.table(T_NEC_BRONZE)
    .filter((F.col("ok") == True) & (F.col("prompt") == PROMPT_FINGERPRINT))
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("llave").orderBy(F.col("ts").desc())))
    .filter(F.col("_rn") == 1)
    .select("no_nec", "llave", "json")
    .collect())

print(f"Bronze OK (dedup): {len(bronze_rows)}")

# ── Cargar datos originales de necropsias (para persona) ──
nec_orig = (spark.table(T_NECROPSIAS)
            .select("no_nec", "sexo_norm", "edad", "fecha_ingreso", "tipo_ingreso", "status")
            .toPandas())

# ── Aplanar ──
vestimenta, tatuajes, pertenencias, persona = [], [], [], []

for r in bronze_rows:
    fb = r.asDict()
    nec = NecropsiaExtraida.model_validate_json(fb["json"])
    no_nec = int(fb["no_nec"])
    
    for i, p in enumerate(nec.prendas):
        d = p.model_dump()
        d["no_nec"] = no_nec
        d["n"] = i
        vestimenta.append(d)
    
    for i, t in enumerate(nec.tatuajes):
        d = t.model_dump()
        d["no_nec"] = no_nec
        d["n"] = i
        tatuajes.append(d)
    
    for i, per in enumerate(nec.pertenencias):
        d = per.model_dump()
        d["no_nec"] = no_nec
        d["n"] = i
        pertenencias.append(d)
    
    # Persona: unir con datos originales
    orig = nec_orig[nec_orig["no_nec"] == no_nec]
    if len(orig):
        edad_min, edad_max = convertir_edad(orig.iloc[0]["edad"])
    else:
        edad_min, edad_max = None, None
    persona.append({
        "no_nec": no_nec,
        "sexo_norm": orig.iloc[0]["sexo_norm"] if len(orig) else None,
        "edad_min": edad_min,
        "edad_max": edad_max,
        "fecha_ingreso": orig.iloc[0]["fecha_ingreso"] if len(orig) else None,
        "tipo_ingreso": orig.iloc[0]["tipo_ingreso"] if len(orig) else None,
        "status": orig.iloc[0]["status"] if len(orig) else None,
        "n_prendas": len(nec.prendas),
        "n_tatuajes": len(nec.tatuajes),
        "n_senas": len(nec.senas),
        "n_pertenencias": len(nec.pertenencias),
    })

# ── Guardar tablas silver ──
_escribir_tabla_pdf(pd.DataFrame(vestimenta), T_NEC_VESTIMENTA)
_escribir_tabla_pdf(pd.DataFrame(tatuajes), T_NEC_TATUAJES)
_escribir_tabla_pdf(pd.DataFrame(pertenencias), T_NEC_PERTENENCIAS)

# Persona (convertir edad a Int32 para Spark)
if len(persona) > 0:
    df_persona = pd.DataFrame(persona)
    for c in ["edad_min", "edad_max"]:
        df_persona[c] = df_persona[c].astype("Int32")
    _escribir_tabla_pdf(df_persona, T_NEC_PERSONA)

print(f"Vestimenta: {len(vestimenta)} | Tatuajes: {len(tatuajes)} | Pertenencias: {len(pertenencias)} | Persona: {len(persona)}")

## 6. Texto canónico por faceta (insumo para embeddings)

Se construye de forma **determinista** desde los campos estructurados (no es texto libre del LLM),
omitiendo `SIN_DATO` y cualquier identificador. Las funciones `texto_vestimenta`, `texto_tatuajes`
y `texto_senas` se copian tal cual del notebook 01; se agrega `texto_pertenencias`.

In [0]:
from pyspark.sql import Window

# ── Funciones de texto canónico copiadas del notebook 01 (sin cambios) ──
def _v(x):
    return None if x in (None, "", SIN_DATO, "sin_dato") else str(x).lower().replace("_", " ")


def texto_tatuajes(rows):
    out = []
    for t in rows:
        s = f"tatuaje en {_v(t['region']) or 'región no especificada'}"
        if _v(t["lado"]):
            s += f" {_v(t['lado'])}"
        s += f": {t['motivo'].lower()}"
        if t.get("texto_legible"):
            s += f", texto '{t['texto_legible']}'"
        if _v(t["color_tinta"]):
            s += f", tinta {_v(t['color_tinta'])}"
        out.append(s)
    return "; ".join(out)


def texto_senas(rows):
    return "; ".join(
        f"{_v(s['tipo'])} en {_v(s['region']) or 'región no especificada'}"
        + (f" {_v(s['lado'])}" if _v(s["lado"]) else "") + f": {s['descripcion'].lower()}"
        for s in rows)


def texto_vestimenta(rows):
    return "; ".join(
        " ".join(filter(None, [_v(r["tipo"]), _v(r["color"]), (r.get("descripcion") or "").lower() or None,
                               f"marca {r['marca']}" if r.get("marca") else None]))
        for r in rows)


# ── Nueva función para pertenencias (sigue el mismo patrón) ──
def texto_pertenencias(rows):
    return "; ".join(
        " ".join(filter(None, [_v(r["tipo"]), (r.get("descripcion") or "").lower() or None,
                               f"marca {r['marca']}" if r.get("marca") else None]))
        for r in rows)


# ── Cargar bronze OK (dedup) y construir texto ──
bronze_ok = (spark.table(T_NEC_BRONZE)
    .filter((F.col("ok") == True) & (F.col("prompt") == PROMPT_FINGERPRINT))
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("llave").orderBy(F.col("ts").desc())))
    .filter(F.col("_rn") == 1)
    .select("no_nec", "json")
    .collect())

filas_texto = []
for r in bronze_ok:
    fb = r.asDict()
    nec = NecropsiaExtraida.model_validate_json(fb["json"])
    no_nec = int(fb["no_nec"])
    prendas = [p.model_dump() for p in nec.prendas]
    tats = [t.model_dump() for t in nec.tatuajes]
    senas = [s.model_dump() for s in nec.senas]
    perts = [p.model_dump() for p in nec.pertenencias]
    filas_texto.append({
        "no_nec": no_nec,
        "texto_vestimenta": texto_vestimenta(prendas),
        "texto_tatuajes": texto_tatuajes(tats),
        "texto_senas": texto_senas(senas),
        "texto_pertenencias": texto_pertenencias(perts),
    })

df_texto = pd.DataFrame(filas_texto)
if len(df_texto):
    df_texto["texto_completo"] = df_texto[["texto_vestimenta", "texto_tatuajes", "texto_senas", "texto_pertenencias"]].apply(
        lambda r: " | ".join(x for x in r if x), axis=1)
    _escribir_tabla_pdf(df_texto, T_NEC_TEXTO)
else:
    print("Sin filas para texto canónico")

## 7. Control de calidad y MLflow

Solo agregados: filas procesadas, % OK, errores por tipo, % con ≥1 prenda,
distribución de tipo y color de prenda (conteos), % con edad convertida.
Se registra un run en MLflow con `corrida("extraccion_necropsias", familia="datos", fase="datos")`.

In [0]:
import mlflow
from collections import Counter

# ── Control de calidad (solo agregados) ──
bronze_all = spark.table(T_NEC_BRONZE).filter(F.col("prompt") == PROMPT_FINGERPRINT)
n_total = bronze_all.count()
n_ok = bronze_all.filter(F.col("ok") == True).count()
n_err = n_total - n_ok
pct_ok = 100.0 * n_ok / max(n_total, 1)

# Errores por tipo
errores_df = bronze_all.filter(F.col("ok") == False).select("error").toPandas()
tipos_error = Counter(e.split(":")[0] for e in errores_df["error"].dropna()) if len(errores_df) else {}

# Persona: % con ≥1 prenda, % con edad convertida
persona_df = spark.table(T_NEC_PERSONA).toPandas() if spark.catalog.tableExists(T_NEC_PERSONA) else pd.DataFrame()
n_con_prenda = int((persona_df["n_prendas"] >= 1).sum()) if len(persona_df) else 0
pct_con_prenda = 100.0 * n_con_prenda / max(len(persona_df), 1)
n_con_edad = int(persona_df["edad_min"].notna().sum()) if len(persona_df) else 0
pct_con_edad = 100.0 * n_con_edad / max(len(persona_df), 1)

# Distribución de tipo y color de prenda (conteos)
vest_df = spark.table(T_NEC_VESTIMENTA).toPandas() if spark.catalog.tableExists(T_NEC_VESTIMENTA) else pd.DataFrame()
dist_tipo = vest_df["tipo"].value_counts().to_dict() if len(vest_df) else {}
dist_color = vest_df["color"].value_counts().to_dict() if len(vest_df) else {}

print(f"Filas procesadas: {n_total}")
print(f"% OK: {pct_ok:.1f}%")
print(f"Errores por tipo: {dict(tipos_error)}")
print(f"% con ≥1 prenda: {pct_con_prenda:.1f}%")
print(f"% con edad convertida: {pct_con_edad:.1f}%")
print(f"Tipos de prenda (top 5): {dict(list(sorted(dist_tipo.items(), key=lambda x: -x[1]))[:5])}")
print(f"Colores de prenda (top 5): {dict(list(sorted(dist_color.items(), key=lambda x: -x[1]))[:5])}")

# ── MLflow ──
n_tatuajes = spark.table(T_NEC_TATUAJES).count() if spark.catalog.tableExists(T_NEC_TATUAJES) else 0
n_pertenencias = spark.table(T_NEC_PERTENENCIAS).count() if spark.catalog.tableExists(T_NEC_PERTENENCIAS) else 0

with corrida("extraccion_necropsias", familia="datos", fase="datos",
             params={"endpoint": ENDPOINT, "prompt_fingerprint": PROMPT_FINGERPRINT,
                     "limite": LIMITE, "max_workers": MAX_WORKERS, "strict_schema": str(STRICT_SCHEMA)}):
    mlflow.log_metric("filas_procesadas", n_total)
    mlflow.log_metric("pct_ok", pct_ok)
    mlflow.log_metric("n_ok", n_ok)
    mlflow.log_metric("n_errores", n_err)
    mlflow.log_metric("pct_con_prenda", pct_con_prenda)
    mlflow.log_metric("pct_con_edad", pct_con_edad)
    mlflow.log_metric("n_prendas_total", len(vest_df))
    mlflow.log_metric("n_tatuajes_total", n_tatuajes)
    mlflow.log_metric("n_pertenencias_total", n_pertenencias)
    mlflow.log_metric("n_persona", len(persona_df))
    for tipo, n in tipos_error.items():
        mlflow.log_metric(f"error_{tipo}", n)
    for tipo, n in list(dist_tipo.items())[:10]:
        mlflow.log_metric(f"prenda_tipo_{tipo}", n)

print("MLflow registrado")